In [1]:
from timescale import TimescaleDBManager
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pandas as pd
import os

In [2]:
def obtener_datos_ejecucion(execution_name, engine,estandarizar=True):
    """
    Extrae todos los registros de una ejecución y métrica específica.
    """
    if estandarizar:
        SQL = """
            SELECT
                m.time,m.metric_name,m.instance,m.grupo,m.job,m.tags,m.label,
                (m.metric_value - e.media) / NULLIF(e.desv_std, 0) AS metric_value
            FROM metricas_transformed m
            INNER JOIN t_escalado e
                ON e.tabla = 'metricas_transformed'
               AND e.execution_name = m.execution_name
               AND e.metric_name = m.metric_name
            
            WHERE m.execution_name = %s 
            ORDER BY time ASC, metric_name ASC, instance ASC, grupo ASC,job ASC

        """
    else:
        SQL = """
            SELECT time,metric_name,instance,grupo,job,tags,label,metric_value
            FROM metricas_transformed
            WHERE execution_name = %s 
            ORDER BY time ASC, metric_name ASC, instance ASC, grupo ASC,job ASC
            
        """
    
    # Ejecutamos la consulta pasando los parámetros de forma segura
    df = pd.read_sql(SQL, engine, params=(execution_name,))
    
    # Convertimos la columna time a formato datetime
    df['time'] = pd.to_datetime(df['time'])
    df         = df.sort_values('time')

    inicio_global = df['time'].min()
    df['tiempo_relativo'] = (df['time'] - inicio_global).dt.total_seconds()
        
    return df

In [3]:
def procesar_datos(lista_de_datos,clave_label,clave_vector):
    lista_result=[]
    if lista_de_datos:
        set_valores = {d[clave_label] for d in lista_de_datos}
        if len(set_valores) == 1:
            label_iguales=True
            valor=set_valores.pop()
            for item in lista_de_datos:
                lista_result.extend(item[clave_vector])
        else:
            #hay valores distintos
            label_iguales=False
            valor=None
    else:
        label_iguales=False
        valor=None
    return label_iguales,valor,lista_result

In [4]:
load_dotenv()
user_env        = os.getenv("USER_DB")
pass_env        = os.getenv("PASS_DB")
host_env        = os.getenv("HOST_DB", "localhost")
port_env        = os.getenv("DB_PORT_EXPOSED", "5432")
db_env          = os.getenv("NAME_DB", "tfm_db")
FOLDER_SALIDA   = os.getenv("FOLDER_SALIDA")
FOLDER_TMP      = os.getenv("FOLDER_TMP")

SEED            = os.getenv("SEED")

db = TimescaleDBManager(
    user=user_env,
    password=pass_env,
    host=host_env,
    port=port_env,
    dbname=db_env
)
db.conectar()
conexion_postgresql = db.connection

#esto es para usar una conexion postgrews para pandas que da un warning indicando que solo esta testeado para sqlalchemy
engine = create_engine(f'postgresql+psycopg2://{user_env}:{pass_env}@{host_env}:{port_env}/{db_env}')

🔌 [OK] Conexión establecida con TimescaleDB.


In [11]:
SQL='select distinct(execution_name) from metricas_transformed'
df_execution_name = pd.read_sql(SQL, engine)
lista_execution_name=df_execution_name['execution_name'].to_list()

In [6]:
df_ejecuciones_outlier = pd.read_parquet(f'{FOLDER_SALIDA}/df_ejecuciones_outlier.parquet')
df_outliers_pca = pd.read_parquet(f'{FOLDER_SALIDA}/df_outliers_pca.parquet')

In [7]:
lista_ejecuciones_outlier = df_ejecuciones_outlier['execution_name'].tolist()

In [8]:
SQL='select distinct(metric_name) from metricas_transformed'
df_metric_name = pd.read_sql(SQL, engine)
lista_metric_name_ord=sorted(df_metric_name['metric_name'].to_list())

In [9]:
len(lista_metric_name_ord)


118

In [12]:
una_ejecucion=lista_execution_name[0]

In [13]:
una_ejecucion

'light-oauth2-data-1719592986.tar'

In [15]:
N=0

In [ ]:
df = obtener_datos_ejecucion(una_ejecucion, engine)
print(f'datos obtenidos execution_name={una_ejecucion}')
N+=1

lista_tiempo_relativo=df['tiempo_relativo'].unique().tolist()
#eliminamos el primer punto, esto es porque las metricas diferenciadas este punto se relleno con 0, evitamos asi usar un valor rellenado
if lista_tiempo_relativo[0]==0.0:
    lista_tiempo_relativo.pop(0)

lista_reg=[]
while len(lista_tiempo_relativo)>=3:
    lista_t=lista_tiempo_relativo[0:3]
    lista_data=[]
    for t_relativo in lista_t:
        data={}
        #df_t = df[df['tiempo_relativo']==t_relativo]
        df_t = df[df['tiempo_relativo']==t_relativo].sort_values(by=['metric_name', 'instance', 'grupo','job'])
        lista_label=df_t['label'].unique().tolist()
        
        if len(lista_label)>1: #esto no deberia pasar, todos los label de este df deberian ser iguales
            break
        label         = lista_label[0]
        data['label'] = label
    
        vector         = df_t['metric_value'].to_list()
        data['vector'] = vector
        lista_data.append(data)
        
    print(f'len(lista_data)={len(lista_data)}')
    valores_iguales,label,lista_metric_value=procesar_datos(lista_data,'label','vector')
    if valores_iguales:
        reg={}
        reg['execution_name'] = una_ejecucion
        reg['t_rel']          = lista_t[0]
        reg['label']          = label
        reg['vector']         = vector

        #almacenar reg
        lista_reg.append(reg)
    lista_tiempo_relativo.pop(0)

In [45]:
valores_iguales,label,vector_unificado=procesar_datos(lista_data,'label','vector')

In [46]:
len(vector_unificado)

354

In [32]:
label

'update_password_400_not_match'

In [34]:
lista_data[0]['label']

'update_password_400_not_match'

In [36]:
lista_result=[]

In [37]:
item=lista_data[0]
lista_result.extend(item['vector'])

In [40]:
item=lista_data[1]
lista_result.extend(item['vector'])

In [43]:
item=lista_data[2]
lista_result.extend(item['vector'])

In [44]:
len(lista_result)

354

In [42]:
lista_result.extend(item[clave_vector])

NameError: name 'clave_vector' is not defined